# NB_FLUX — repaired validation notebook

This clean notebook replaces the broken upload/rename workflow in `NB_FLUX_untitled238.ipynb`.

**Purpose:** validate the exact finite-torus carrier checks, reproduce the published Wilson-action regression, and run a tiny SU(3) Monte Carlo smoke test after repairing the APE gauge-covariance defect.

**Scope:** validation only. The smoke mass and amplitude ratio are not physics evidence. Pilot/continuum production is deliberately not launched here.

Key repair: the original ordered Gram–Schmidt “SU(3) projection” was not gauge covariant. The companion engine now uses the polar/SVD projection and tests both the projection and the complete APE map under random local gauge transformations.

Original SHA-256: `EB0E4D364976692289AE24FE25021898BCB015FD86E5E2EC159420A4F01074E5`  
Engine SHA-256: `258FEFD36B4155D1E96376A7FFCC4282B810BD2F1279B90DD58A8727FABCB42C`


In [1]:
from pathlib import Path
import hashlib
import importlib.util
import json
import sys
import scipy

ENGINE_PATH = Path("NB_FLUX_T1PM_engine_repaired.py").resolve()
EXPECTED_SHA256 = "258FEFD36B4155D1E96376A7FFCC4282B810BD2F1279B90DD58A8727FABCB42C"
if not ENGINE_PATH.is_file():
    raise FileNotFoundError(
        "Keep this notebook and NB_FLUX_T1PM_engine_repaired.py in the same directory."
    )
actual_sha = hashlib.sha256(ENGINE_PATH.read_bytes()).hexdigest().upper()
if actual_sha != EXPECTED_SHA256:
    raise RuntimeError(f"Engine hash mismatch: {actual_sha} != {EXPECTED_SHA256}")

spec = importlib.util.spec_from_file_location("nb_flux_engine", ENGINE_PATH)
engine = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = engine
spec.loader.exec_module(engine)

print(engine.ENGINE_VERSION)
print("Engine hash verified:", actual_sha)
print("NumPy:", engine.np.__version__)
print("SciPy:", scipy.__version__)
reference_versions = {"numpy": "2.3.5", "scipy": "1.18.1"}
current_versions = {"numpy": engine.np.__version__, "scipy": scipy.__version__}
if current_versions != reference_versions:
    print("WARNING: exact seeded trajectory requires NB_FLUX_REQUIREMENTS_LOCK.txt", current_versions)


NB-FLUX repaired 2026-08-22
Engine hash verified: 258FEFD36B4155D1E96376A7FFCC4282B810BD2F1279B90DD58A8727FABCB42C
NumPy: 2.3.5
SciPy: 1.18.1


## 1. Deterministic certificate and published replay

This phase contains no new Monte Carlo evidence. The chain ranks are computed exactly modulo two primes; the harmonic carrier representation and the P/R/S loop transformation rules are evaluated rather than hard-coded. The published continuum numbers are a transcription/regression test against Athenodorou–Teper, not an independent calculation.


In [2]:
engine.main(["--profile", "replay", "--no-gpu"])
assert all(g.passed for g in engine.GATES if g.hard)
print("\nDeterministic/replay hard gates:", sum(g.hard and g.passed for g in engine.GATES), "/", sum(g.hard for g in engine.GATES))


Spatial SU(3) T1^{+-} cubic-Casimir bridge test (NB-FLUX repaired 2026-08-22)
profile=replay, seed=20260801

A. EXACT SPATIAL CARRIER CERTIFICATE
  [PASS] chain condition d2 d3 = 0: max exact entry=0
  [PASS] three harmonic plaquette planes: d2 H max=0, d3^T H max=0, b2=3
  [PASS] torus incidence ranks (exact modular certificate): rank(d2)=(126, 126), rank(d3)=(63, 63) over primes (1000003, 1000033)
  [PASS] translated cube boundaries have no k=0 carrier: max entry of d3*1=0
  [PASS] 24 proper cubic rotations: signed-permutation group O generated exactly
  [PASS] cube-boundary irrep: proper-rotation scalar and parity odd; with ImTr it is A1^{--}, not T1^{+-}
  [PASS] zero-momentum harmonic carrier representation: actual cochain permutation gives P_R H=H R for all 24 rotations and P H=H; ImTr is C-odd
  [PASS] measured-loop RPC projector: all P/R/S paths pass 24 rotations, P=+, and C=- exactly

C. PUBLISHED WILSON-ACTION CONTINUUM REPLAY
  beta       a^2 sigma      aM(T1+-)       M/sqrt

## 2. Repaired Monte Carlo smoke test

The following run is intentionally tiny (`4^3 × 8`, 48 measurements). It checks execution, SU(3) integrity, post-warm-up action/staple consistency, global over-relaxation invariance, projection covariance, and full APE-step local gauge covariance.

It is expected to warn about autocorrelation coverage, bootstrap blocks, and finite volume. Those warnings are the reason its fitted mass is **not** a scientific result.

The executable topology certificate is evaluated on the `L=4` periodic complex. The exploratory T1 fit is a soft execution diagnostic; in the pinned smoke run only 35 of 60 bootstrap fits succeeded.


In [3]:
RESULT_PATH = Path("NB_FLUX_T1PM_repaired_smoke_result.json")
engine.main([
    "--profile", "smoke",
    "--no-gpu",
    "--seed", "20260801",
    "--json", str(RESULT_PATH),
])
assert RESULT_PATH.is_file()
result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
assert result["overall_passed"] is True
print("\nStrict result JSON:", RESULT_PATH.resolve())


Spatial SU(3) T1^{+-} cubic-Casimir bridge test (NB-FLUX repaired 2026-08-22)
profile=smoke, seed=20260801

A. EXACT SPATIAL CARRIER CERTIFICATE
  [PASS] chain condition d2 d3 = 0: max exact entry=0
  [PASS] three harmonic plaquette planes: d2 H max=0, d3^T H max=0, b2=3
  [PASS] torus incidence ranks (exact modular certificate): rank(d2)=(126, 126), rank(d3)=(63, 63) over primes (1000003, 1000033)
  [PASS] translated cube boundaries have no k=0 carrier: max entry of d3*1=0
  [PASS] 24 proper cubic rotations: signed-permutation group O generated exactly
  [PASS] cube-boundary irrep: proper-rotation scalar and parity odd; with ImTr it is A1^{--}, not T1^{+-}
  [PASS] zero-momentum harmonic carrier representation: actual cochain permutation gives P_R H=H R for all 24 rotations and P H=H; ImTr is C-odd
  [PASS] measured-loop RPC projector: all P/R/S paths pass 24 rotations, P=+, and C=- exactly

C. PUBLISHED WILSON-ACTION CONTINUUM REPLAY
  beta       a^2 sigma      aM(T1+-)       M/sqrt(

  cycle      5/40: plaquette=0.79621732, accept=0.7661, proposal=0.3292


  cycle     10/40: plaquette=0.70772863, accept=0.7491, proposal=0.3584


  cycle     15/40: plaquette=0.65838209, accept=0.7333, proposal=0.3875
  cycle     20/40: plaquette=0.62360266, accept=0.7192, proposal=0.4162


  cycle     25/40: plaquette=0.60472929, accept=0.7017, proposal=0.4437
  cycle     30/40: plaquette=0.59549501, accept=0.6859, proposal=0.4695


...

  cfg    32/48: plaquette=0.55577608, accept=0.6528, elapsed=2.5s


  cfg    36/48: plaquette=0.56420696, accept=0.6540, elapsed=2.8s


  cfg    40/48: plaquette=0.56310025, accept=0.6496, elapsed=3.1s


  cfg    44/48: plaquette=0.56029979, accept=0.6429, elapsed=3.4s


  cfg    48/48: plaquette=0.56416014, accept=0.6562, elapsed=3.7s
  [WARN] smoke autocorrelation coverage: estimated 2*tau=15.43 exceeds smoke block=12; no physics error claim

  autocorrelation: tau_plaq=7.71, tau_O2=0.50 measurements; block=12, blocks=4
  [WARN] bootstrap block count: 4 blocks of 12 measurements

Spectroscopy result
-------------------
  T1^{+-}: aM=0.487566 +/- 0.508598, fit [1,3], chi2/dof=0.39/1, p=0.535
  torelon:  aE=0.347350 +/- 0.089441 -> a sqrt(sigma)=0.417361 +/- 0.021201
  M/sqrt(sigma)=1.168213 +/- 1.220048
  raw ImTr(plaquette) exploratory fitted amplitude ratio=0.039109 +/- 0.036624
  (diagnostic only: this is not a normalized spectral residue)
  variational ranks: T1=4 (cond 3.49e+02), torelon=2 (cond 5.40e+01)
  [PASS] T1 exploratory mass fit: aM=0.4876, dof=1, p=0.535, bootstrap=35/60
  [PASS] torelon exploratory fit: aE=0.3473, dof=1, p=0.944, bootstrap=60/60
  [WARN] finite-volume scale: L a sqrt(sigma)=1.669 (target >=3)
  [INFO] no pass/fail gate

In [4]:
failed_hard = [g for g in result["gates"] if g["hard"] and not g["passed"]]
warnings = [g for g in result["gates"] if (not g["hard"]) and not g["passed"]]
t1 = result["ensemble"]["t1"]

summary = {
    "hard_failures": [g["name"] for g in failed_hard],
    "warnings": [g["name"] for g in warnings],
    "APE_projection_gate": next(g for g in result["gates"] if g["name"] == "APE SU(3) projection gauge covariance")["detail"],
    "full_APE_gate": next(g for g in result["gates"] if g["name"].startswith("full APE-step"))["detail"],
    "exploratory_aM": t1["mass"],
    "exploratory_aM_error": t1["mass_error"],
    "bootstrap_success_fraction": t1["bootstrap_success_fraction"],
    "raw_amplitude_ratio_not_residue": t1["raw_amplitude_ratio"],
}
summary


{'hard_failures': [],
 'warnings': ['smoke autocorrelation coverage',
  'bootstrap block count',
  'finite-volume scale'],
 'APE_projection_gate': 'covariance=3.118e-07, unitarity=3.167e-07, det=2.298e-07',
 'full_APE_gate': 'max transformed-link Frobenius error=4.553e-07',
 'exploratory_aM': 0.48756633721694326,
 'exploratory_aM_error': 0.5085975682723671,
 'bootstrap_success_fraction': 0.5833333333333334,
 'raw_amplitude_ratio_not_residue': 0.039109320494403896}

## 3. Run decision

The repaired smoke workflow is worth keeping and rerunning as a regression. Do **not** launch the supplied pilot or six-volume continuum profiles as physics runs yet.

The next scientifically useful notebook must add:

1. raw and improved charge-odd sources at a fixed physical dressing radius;
2. separate continuum-tensor sources `V_T1`, `H_T1`, `H_A2`, and `H_T2` to resolve the leading `J=1 ⊕ J=3` source mixture;
3. contaminant operators, multiple volumes, topology monitoring, several independent chains, and checkpoints;
4. shared blocked resampling and correlated fit/basis/window stability.

The A100 is useful after that redesign. This CuPy implementation does not use the local AMD 7900 XTX; that would require a ROCm-capable backend.
